# 01 — EDA: price vs demand

Goal: understand the price–demand relationship before modeling. We work in logs because
constant-elasticity demand `q = A·p^ε` is linear in `log(q) ~ log(p)`, and the slope **is** the elasticity.

Dataset: Kaggle — Retail Price Optimization (Suddharshan). A product×month panel (52 products, 20 months, 9 categories).
Fetch per the README; nothing is committed.

Watch for **price endogeneity**: if stores raise prices when demand is high, the raw pooled slope
is biased toward zero. The markdown proxy and the within-product views below start to expose that.

In [ ]:
import sys; sys.path.insert(0, "..")  # import the project's src/ package
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.data import load_prepared

pd.set_option("display.max_columns", 50)
df = load_prepared()
print(df.shape)
df.head()

## log–log scatter
Plot `log(quantity)` vs `log(price)`, colored by the markdown proxy. A clean negative slope
is the elasticity; a flat/positive cloud is endogeneity (price set in response to demand) showing itself.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(data=df, x="log_p", y="log_q", hue="is_markdown", alpha=0.4, ax=ax)
ax.set(title="log demand vs log price", xlabel="log(unit_price)", ylabel="log(qty)")
plt.show()

## within-product view
The pooled cloud mixes cheap and expensive products. The real signal is *within* a product over time —
exactly what product fixed effects isolate in the next notebook. Spot-check a few high-coverage products:

In [ ]:
top = df["product_id"].value_counts().head(6).index
g = sns.lmplot(data=df[df.product_id.isin(top)], x="log_p", y="log_q",
               col="product_id", col_wrap=3, height=2.6, scatter_kws={"alpha": 0.5})
g.set_titles("{col_name}")
plt.show()

### Next: `02_elasticity.ipynb`
Naive OLS log-log elasticity (the biased foil), then the causal correction
(product/time fixed effects, then IV with lagged + competitor prices).